# InvariantRRF v4.3 — SPLADE Two-Checkpoint Closure Audit

This notebook performs the provenance-clean rerun of the SPLADE checkpoint-family boundary experiment.

It imports the frozen v4.3 canonical archive, reuses its exact BM25 and SPLADE-EnsembleDistil rankings, generates only `naver/splade-cocondenser-selfdistil`, reruns the depth-20/50/100 experiment, recomputes the four planned Holm-corrected tests at depth 50, and compares every paper-facing value against the current v4.3 manuscript values.

No BM25 regeneration. No EnsembleDistil regeneration. Qrels are used only for nDCG evaluation, never for family definition.


In [ ]:
from pathlib import Path
from collections import defaultdict
import gc
import hashlib
import json
import math
import random
import shutil
import subprocess
import sys
import time
import warnings
import zipfile

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

try:
    from scipy.stats import wilcoxon, rankdata
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy.stats import wilcoxon, rankdata

try:
    from sentence_transformers import SparseEncoder
    from sentence_transformers.util import semantic_search, dot_score
except Exception:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "sentence-transformers"]
    )
    from sentence_transformers import SparseEncoder
    from sentence_transformers.util import semantic_search, dot_score


# ============================================================
# 0. USER CONFIGURATION
# ============================================================

# Use the same frozen v4.3 archive used by the partial-observation closure audit.
# Leave None to auto-discover the highest-priority InvariantRRF ZIP under
# /kaggle/input, /kaggle/working, /content, or the current directory.
INPUT_ZIP = None

# Optional alternative: path to an already extracted canonical directory
# containing runs/ and ideally RUN_MANIFEST.json.
CANON_OVERRIDE = None

SEED = 20260903
random.seed(SEED)
np.random.seed(SEED)

DATASETS_TO_RUN = ["SciFact", "ArguAna"]
DATASET_DIRNAMES = {
    "SciFact": "scifact_mpdr",
    "ArguAna": "arguana_mpdr",
}
EXPECTED_Q = {
    "SciFact": 300,
    "ArguAna": 1401,
}

RRF_K = 60.0
EVAL_K = 10
DEPTHS = [20, 50, 100]
PRIMARY_DEPTH = 50
RBO_P = 0.90
N_BOOT = 10_000
ALPHA = 0.05

SELF_MODEL = "naver/splade-cocondenser-selfdistil"
SELF_TOPK = 500
SPLADE_BATCH_SIZE = 32
SPLADE_CORPUS_CHUNK = 50_000

WORK_ROOT = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
    if Path("/content").exists()
    else Path.cwd()
)

OUT = WORK_ROOT / "InvariantRRF_V43_SPLADE_TwoCheckpoint_Closure"
RUN_OUT = OUT / "generated_runs"

if OUT.exists():
    shutil.rmtree(OUT)
RUN_OUT.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT:", WORK_ROOT)
print("OUT:", OUT)
print("SelfDistil model:", SELF_MODEL)


# ============================================================
# 1. IMPORT AND VERIFY THE FROZEN V4.3 CANONICAL ARCHIVE
# ============================================================

def sha256_file(path, chunk=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def find_canonical_root(root):
    root = Path(root)

    if (root / "runs").is_dir():
        return root

    for manifest in sorted(root.rglob("RUN_MANIFEST.json")):
        parent = manifest.parent
        if (parent / "runs").is_dir():
            return parent

    for runs_dir in sorted(p for p in root.rglob("runs") if p.is_dir()):
        return runs_dir.parent

    raise FileNotFoundError(
        f"Could not find a canonical directory containing runs/ under {root}"
    )


def discover_canonical_zip():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/content"), Path.cwd()]
    candidates = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob("*.zip"):
                rp = str(p.resolve())
                if rp in seen:
                    continue
                seen.add(rp)
                name = p.name.lower()
                score = (100 if "invariantrrf" in name else 0) + (50 if "deepdense" in name else 0) + (40 if "taskadaptivek" in name else 0) + (30 if "canonical" in name else 0) + (20 if "strict" in name else 0)
                candidates.append((score, str(p), p))
        except Exception:
            pass
    if not candidates:
        raise FileNotFoundError("No canonical InvariantRRF ZIP found. Set INPUT_ZIP explicitly.")
    candidates.sort(reverse=True)
    print("Canonical ZIP candidates:")
    for score, _, p in candidates[:10]:
        print(" ", p, "priority=", score)
    return candidates[0][2]


if CANON_OVERRIDE is not None:
    CANON = find_canonical_root(Path(CANON_OVERRIDE))
    SOURCE_ARCHIVE = None
else:
    archive = Path(INPUT_ZIP) if INPUT_ZIP else discover_canonical_zip()
    if not archive.exists():
        raise FileNotFoundError(f"Frozen v4.3 archive not found: {archive}")

    SOURCE_ARCHIVE = archive
    EXTRACT_ROOT = WORK_ROOT / "_v43_splade_closure_import"

    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(EXTRACT_ROOT)

    CANON = find_canonical_root(EXTRACT_ROOT)

CANON_RUNS = CANON / "runs"
assert CANON_RUNS.is_dir()

print("\nSOURCE_ARCHIVE:", SOURCE_ARCHIVE)
print("CANON:", CANON)
print("CANON_RUNS:", CANON_RUNS)

manifest_path = CANON / "RUN_MANIFEST.json"

if manifest_path.exists():
    original_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_files = original_manifest.get("files", {})

    failures = []
    checked = 0

    for rel, expected in manifest_files.items():
        p = CANON / rel

        if not p.exists():
            failures.append((rel, "MISSING", expected))
            continue

        actual = sha256_file(p)
        checked += 1

        if actual != expected:
            failures.append((rel, actual, expected))

    if failures:
        print("Manifest failures:")
        for row in failures[:20]:
            print(row)
        raise AssertionError(
            f"Frozen v4.3 manifest verification failed for {len(failures)} file(s)."
        )

    print(f"ORIGINAL V4.3 MANIFEST: PASS ({checked} files verified)")
else:
    original_manifest = None
    print(
        "WARNING: RUN_MANIFEST.json not found. "
        "Direct source-file hashes will still be recorded."
    )


# ============================================================
# 2. LOAD THE EXACT FROZEN BM25 AND ENSEMBLEDISTIL RUNS
# ============================================================

CANONICAL_SOURCE_PATHS = {
    "SciFact": {
        "bm25": CANON_RUNS / "SciFact_bm25_top1000.json",
        "splade_ensemble": CANON_RUNS / "SciFact_splade_ensemble_top500.json",
    },
    "ArguAna": {
        "bm25": CANON_RUNS / "ArguAna_bm25_top1000.json",
        "splade_ensemble": CANON_RUNS / "ArguAna_splade_ensemble_top500.json",
    },
}


def load_run(path):
    path = Path(path)
    assert path.exists(), f"Missing canonical source: {path}"

    raw = json.loads(path.read_text(encoding="utf-8"))
    assert isinstance(raw, dict) and raw

    out = {}
    for qid, vals in raw.items():
        seq = []
        for x in vals:
            if isinstance(x, (list, tuple)):
                seq.append((str(x[0]), float(x[1])))
            else:
                seq.append((str(x), 0.0))
        out[str(qid)] = seq

    return out


def docids(seq):
    ids = [str(d) for d, _ in seq]
    if len(ids) != len(set(ids)):
        raise AssertionError("Duplicate document ID in ranked run.")
    return ids


CANONICAL_RUNS = {}
source_hashes = {}
source_rows = []

for ds_name, paths in CANONICAL_SOURCE_PATHS.items():
    CANONICAL_RUNS[ds_name] = {}

    for source_name, path in paths.items():
        run = load_run(path)
        CANONICAL_RUNS[ds_name][source_name] = run

        required_depth = 1000 if source_name == "bm25" else 500

        assert len(run) == EXPECTED_Q[ds_name], (
            ds_name,
            source_name,
            len(run),
            EXPECTED_Q[ds_name],
        )

        depths = []
        for qid, seq in run.items():
            ids = docids(seq)
            depths.append(len(ids))
            assert len(ids) >= required_depth, (
                ds_name,
                source_name,
                qid,
                len(ids),
                required_depth,
            )

        relpath = str(path.relative_to(CANON))
        source_hashes[relpath] = sha256_file(path)

        source_rows.append(
            {
                "dataset": ds_name,
                "source": source_name,
                "queries": len(run),
                "min_depth": min(depths),
                "max_depth": max(depths),
                "sha256": source_hashes[relpath],
            }
        )

source_df = pd.DataFrame(source_rows)
display(source_df)
print("FROZEN CANONICAL BM25 + ENSEMBLEDISTIL SOURCES: PASS")


# ============================================================
# 3. DISCOVER THE EXACT DATASET TEXT/QRELS POPULATIONS
# ============================================================

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_qrels_tsv(root):
    path = root / "dev" / "qrels.tsv"
    qrels = defaultdict(dict)

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            sp = line.rstrip("\n").split("\t")
            if len(sp) < 2:
                continue

            try:
                rel = float(sp[2]) if len(sp) >= 3 and sp[2] != "" else 1.0
            except ValueError:
                continue

            qid, did = str(sp[0]), str(sp[1])
            if rel > 0:
                qrels[qid][did] = rel

    return dict(qrels)


def find_dataset_root(dirname):
    # Public/GitHub version: discover the prepared benchmark directory recursively.
    base = Path("/kaggle/input")
    if base.exists():
        for p in base.rglob(dirname):
            if p.is_dir() and all(
                (p / "dev" / fn).exists()
                for fn in ["queries.jsonl", "docs.jsonl", "qrels.tsv"]
            ):
                return p

    return None


DS = {}

for ds_name in DATASETS_TO_RUN:
    root = find_dataset_root(DATASET_DIRNAMES[ds_name])

    if root is None:
        raise FileNotFoundError(
            f"Could not find {DATASET_DIRNAMES[ds_name]} under /kaggle/input. "
            "Attach the same benchmark source used by the v4.3 rerun."
        )

    queries = read_jsonl(root / "dev" / "queries.jsonl")
    docs = read_jsonl(root / "dev" / "docs.jsonl")
    qrels = load_qrels_tsv(root)

    qmap = {str(x["id"]): x.get("text", "") for x in queries}
    dmap = {str(x["id"]): x.get("text", "") for x in docs}
    qids = sorted(set(qrels) & set(qmap))

    assert len(qids) == EXPECTED_Q[ds_name], (
        ds_name,
        len(qids),
        EXPECTED_Q[ds_name],
    )

    assert set(qids) == set(CANONICAL_RUNS[ds_name]["bm25"]), (
        ds_name,
        "BM25 qid mismatch",
    )

    assert set(qids) == set(CANONICAL_RUNS[ds_name]["splade_ensemble"]), (
        ds_name,
        "EnsembleDistil qid mismatch",
    )

    known_docs = set(dmap)

    for source_name in ["bm25", "splade_ensemble"]:
        for qid in qids:
            ids = docids(CANONICAL_RUNS[ds_name][source_name][qid])
            assert set(ids).issubset(known_docs), (
                ds_name,
                source_name,
                qid,
                "unknown canonical document ID",
            )

    DS[ds_name] = {
        "root": root,
        "qmap": qmap,
        "dmap": dmap,
        "qrels": qrels,
        "qids": qids,
        "runs": {
            "bm25": CANONICAL_RUNS[ds_name]["bm25"],
            "splade_ensemble": CANONICAL_RUNS[ds_name]["splade_ensemble"],
        },
    }

    print(
        ds_name,
        "root=", root,
        "queries=", len(qids),
        "docs=", len(dmap),
    )

print("BENCHMARK ALIGNMENT WITH FROZEN RUNS: PASS")


# ============================================================
# 4. EVALUATION AND STATISTICAL HELPERS
# ============================================================

def dcg_at_k(docids_, rels, k=10):
    gain = 0.0

    for rank, d in enumerate(docids_[:k], 1):
        rel = float(rels.get(str(d), 0.0))
        gain += (2.0 ** rel - 1.0) / math.log2(rank + 1.0)

    return gain


def ndcg_at_k(docids_, rels, k=10):
    dcg = dcg_at_k(docids_, rels, k)

    ideal_rels = sorted(
        (float(v) for v in rels.values()),
        reverse=True,
    )[:k]

    if not ideal_rels:
        return 0.0

    idcg = sum(
        (2.0 ** rel - 1.0) / math.log2(i + 2.0)
        for i, rel in enumerate(ideal_rels)
    )

    return dcg / idcg if idcg > 0 else 0.0


def rbo_finite(a, b, p=0.9, depth=None):
    depth = min(
        depth or max(len(a), len(b)),
        max(len(a), len(b)),
    )

    if depth <= 0:
        return 1.0

    A, B = set(), set()
    num = 0.0
    den = 0.0

    for d in range(1, depth + 1):
        if d <= len(a):
            A.add(a[d - 1])

        if d <= len(b):
            B.add(b[d - 1])

        w = p ** (d - 1)
        num += (len(A & B) / d) * w
        den += w

    return num / den if den > 0 else 1.0


def paired_bootstrap_ci(diffs, B=N_BOOT, seed=SEED, alpha=ALPHA):
    x = np.asarray(diffs, dtype=float)

    if len(x) == 0:
        return (np.nan, np.nan)

    rng = np.random.default_rng(seed)
    n = len(x)
    means = np.empty(B, dtype=float)

    for b in range(B):
        means[b] = x[
            rng.integers(0, n, size=n)
        ].mean()

    return tuple(
        np.quantile(
            means,
            [alpha / 2, 1 - alpha / 2],
        )
    )


def wilcoxon_safe(diffs):
    x = np.asarray(diffs, dtype=float)
    nz = x[np.abs(x) > 1e-15]

    if len(nz) == 0:
        return 1.0

    return float(
        wilcoxon(
            x,
            zero_method="wilcox",
            alternative="two-sided",
            method="auto",
        ).pvalue
    )


def rank_biserial(diffs):
    x = np.asarray(diffs, dtype=float)
    nz = x[np.abs(x) > 1e-15]

    if len(nz) == 0:
        return 0.0

    ranks = rankdata(np.abs(nz), method="average")
    pos = ranks[nz > 0].sum()
    neg = ranks[nz < 0].sum()
    den = pos + neg

    return float(
        (pos - neg) / den
    ) if den > 0 else 0.0


def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    out = np.empty(m, dtype=float)
    running = 0.0

    for j, idx in enumerate(order):
        adj = (m - j) * p[idx]
        running = max(running, adj)
        out[idx] = min(1.0, running)

    return out


# ============================================================
# 5. GENERATE ONLY SELFDISTIL WITH DETERMINISTIC TOTAL ORDERING
# ============================================================

def save_run(path, run):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    serial = {
        str(q): [
            [str(d), float(s)]
            for d, s in vals
        ]
        for q, vals in run.items()
    }

    path.write_text(
        json.dumps(
            serial,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def deterministic_splade_topk(
    Q,
    D,
    qids,
    docids_,
    k,
    semantic_search_fn,
    dot_score_fn,
    corpus_chunk_size=50_000,
):
    # Deterministic SPLADE top-k:
    # 1. retrieve a safe candidate pool;
    # 2. sort by (-score, document_id);
    # 3. if the k-boundary may be tied with unseen candidates,
    #    run a full per-query fallback;
    # 4. if fewer than k positive-score documents exist,
    #    fill zero-score documents by lexicographic document_id.

    k = min(int(k), len(docids_))
    all_docids_sorted = sorted(map(str, docids_))

    retrieve_k = min(
        len(docids_),
        max(k * 2, k + 100),
    )

    hits = semantic_search_fn(
        Q,
        D,
        top_k=retrieve_k,
        score_function=dot_score_fn,
        query_chunk_size=16,
        corpus_chunk_size=corpus_chunk_size,
    )

    run = {}

    for qi, qid in enumerate(qids):
        pairs = [
            (
                str(docids_[int(h["corpus_id"])]),
                float(h["score"]),
            )
            for h in hits[qi]
        ]

        if any(
            not math.isfinite(sc)
            for _, sc in pairs
        ):
            raise RuntimeError(
                f"Non-finite SelfDistil score for query {qid}."
            )

        pairs.sort(
            key=lambda x: (-x[1], x[0])
        )

        positive = [
            x
            for x in pairs
            if x[1] > 0.0
        ]

        if (
            len(positive) >= k
            and retrieve_k < len(docids_)
            and len(pairs) > k
            and pairs[k - 1][1] == pairs[-1][1]
        ):
            full = semantic_search_fn(
                Q[qi : qi + 1],
                D,
                top_k=len(docids_),
                score_function=dot_score_fn,
                query_chunk_size=1,
                corpus_chunk_size=corpus_chunk_size,
            )[0]

            pairs = [
                (
                    str(docids_[int(h["corpus_id"])]),
                    float(h["score"]),
                )
                for h in full
            ]

            if any(
                not math.isfinite(sc)
                for _, sc in pairs
            ):
                raise RuntimeError(
                    f"Non-finite SelfDistil full-fallback score for query {qid}."
                )

            pairs.sort(
                key=lambda x: (-x[1], x[0])
            )

            positive = [
                x
                for x in pairs
                if x[1] > 0.0
            ]

        if len(positive) < k:
            seen = {
                d
                for d, _ in positive
            }

            chosen = positive + [
                (d, 0.0)
                for d in all_docids_sorted
                if d not in seen
            ][: k - len(positive)]

        else:
            chosen = positive[:k]

        if len(chosen) != k:
            raise AssertionError(
                f"{qid}: SelfDistil depth {len(chosen)} != {k}"
            )

        if len(
            {
                d
                for d, _ in chosen
            }
        ) != k:
            raise AssertionError(
                f"{qid}: duplicate SelfDistil document IDs"
            )

        run[str(qid)] = chosen

    del hits
    return run


generation_rows = []
self_run_hashes = {}

for ds_name, data in DS.items():
    out_path = (
        RUN_OUT
        / f"{ds_name}_splade_selfdistil_top500.json"
    )

    print(
        f"\n[{ds_name}] loading {SELF_MODEL}"
    )

    t0 = time.time()

    model = SparseEncoder(SELF_MODEL)

    docids_ = list(data["dmap"])
    qids_ = list(data["qids"])

    docs = [
        data["dmap"][d]
        for d in docids_
    ]

    queries = [
        data["qmap"][q]
        for q in qids_
    ]

    print(
        f"[{ds_name}] encoding {len(docs)} documents"
    )

    D = model.encode_document(
        docs,
        convert_to_sparse_tensor=True,
        batch_size=SPLADE_BATCH_SIZE,
        show_progress_bar=True,
    )

    print(
        f"[{ds_name}] encoding {len(queries)} queries"
    )

    Q = model.encode_query(
        queries,
        convert_to_sparse_tensor=True,
        batch_size=SPLADE_BATCH_SIZE,
        show_progress_bar=True,
    )

    try:
        D = D.cpu()
    except Exception:
        pass

    try:
        Q = Q.cpu()
    except Exception:
        pass

    run = deterministic_splade_topk(
        Q=Q,
        D=D,
        qids=qids_,
        docids_=docids_,
        k=SELF_TOPK,
        semantic_search_fn=semantic_search,
        dot_score_fn=dot_score,
        corpus_chunk_size=SPLADE_CORPUS_CHUNK,
    )

    assert len(run) == EXPECTED_Q[ds_name]
    assert all(
        len(v) == SELF_TOPK
        for v in run.values()
    )

    save_run(
        out_path,
        run,
    )

    run_hash = sha256_file(out_path)

    data["runs"]["splade_self"] = run
    self_run_hashes[out_path.name] = run_hash

    elapsed = time.time() - t0

    generation_rows.append(
        {
            "dataset": ds_name,
            "model": SELF_MODEL,
            "queries": len(run),
            "depth": SELF_TOPK,
            "seconds": elapsed,
            "sha256": run_hash,
        }
    )

    del model, D, Q, run, docs, queries
    gc.collect()

    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception:
        pass


generation_df = pd.DataFrame(
    generation_rows
)

generation_df.to_csv(
    OUT / "selfdistil_generation_log.csv",
    index=False,
)

display(generation_df)
print("SELFDISTIL GENERATION: PASS")


# ============================================================
# 6. RRF AND FAMILY-MASS HELPERS
# ============================================================

def prefix_docs(seq, L):
    return [
        str(d)
        for d, _ in seq[
            : min(int(L), len(seq))
        ]
    ]


def rrf_fuse(
    rankings,
    depth,
    k=60.0,
    weights=None,
):
    if weights is None:
        weights = {
            name: 1.0
            for name in rankings
        }

    else:
        weights = {
            name: float(weights[name])
            for name in rankings
        }

    scores = defaultdict(float)

    for name, seq in rankings.items():
        for rank, did in enumerate(
            prefix_docs(seq, depth),
            1,
        ):
            scores[did] += (
                weights[name]
                / (float(k) + rank)
            )

    return sorted(
        scores.items(),
        key=lambda x: (-x[1], x[0]),
    )


def clean_mc_weights():
    return {
        "bm25": 1.0,
        "splade_ensemble": 1.0,
    }


def attacked_mc_weights():
    return {
        "bm25": 1.0,
        "splade_ensemble": 0.5,
        "splade_self": 0.5,
    }


def score_max_abs_diff(a, b):
    A = dict(a)
    B = dict(b)

    if set(A) != set(B):
        return np.inf

    return max(
        (
            abs(
                float(A[d])
                - float(B[d])
            )
            for d in A
        ),
        default=0.0,
    )


def top_docs(result, k=10):
    return [
        d
        for d, _ in result[:k]
    ]


# ============================================================
# 7. RUN THE TWO-CHECKPOINT FAMILY EXPERIMENT
# ============================================================

summary_rows = []
perq_rows = []
rbo_rows = []

max_clean_reduction_diff = 0.0
max_undergroup_diff = 0.0

for ds_name, data in DS.items():
    required = [
        "bm25",
        "splade_ensemble",
        "splade_self",
    ]

    if not all(
        name in data["runs"]
        for name in required
    ):
        raise RuntimeError(
            f"{ds_name}: missing one of {required}"
        )

    qids = sorted(
        set(data["qids"]).intersection(
            *(
                set(data["runs"][name])
                for name in required
            )
        )
    )

    assert len(qids) == EXPECTED_Q[ds_name]

    print(
        ds_name,
        "aligned queries:",
        len(qids),
    )

    for L in DEPTHS:
        ord_order = []
        ord_set = []
        mc_order = []
        mc_set = []

        nd_clean = []
        nd_ord = []
        nd_mc = []

        rbo_vals = []

        for q in qids:
            clean_rankings = {
                "bm25": data["runs"]["bm25"][q],
                "splade_ensemble": data["runs"]["splade_ensemble"][q],
            }

            attacked_rankings = {
                **clean_rankings,
                "splade_self": data["runs"]["splade_self"][q],
            }

            clean_ord = rrf_fuse(
                clean_rankings,
                L,
                k=RRF_K,
            )

            clean_mc = rrf_fuse(
                clean_rankings,
                L,
                k=RRF_K,
                weights=clean_mc_weights(),
            )

            attacked_ord = rrf_fuse(
                attacked_rankings,
                L,
                k=RRF_K,
            )

            attacked_mc = rrf_fuse(
                attacked_rankings,
                L,
                k=RRF_K,
                weights=attacked_mc_weights(),
            )

            undergroup_mc = rrf_fuse(
                attacked_rankings,
                L,
                k=RRF_K,
                weights={
                    "bm25": 1.0,
                    "splade_ensemble": 1.0,
                    "splade_self": 1.0,
                },
            )

            max_clean_reduction_diff = max(
                max_clean_reduction_diff,
                score_max_abs_diff(
                    clean_ord,
                    clean_mc,
                ),
            )

            max_undergroup_diff = max(
                max_undergroup_diff,
                score_max_abs_diff(
                    attacked_ord,
                    undergroup_mc,
                ),
            )

            c = top_docs(
                clean_ord,
                EVAL_K,
            )

            o = top_docs(
                attacked_ord,
                EVAL_K,
            )

            m = top_docs(
                attacked_mc,
                EVAL_K,
            )

            ord_order.append(
                int(o == c)
            )

            ord_set.append(
                int(set(o) == set(c))
            )

            mc_order.append(
                int(m == c)
            )

            mc_set.append(
                int(set(m) == set(c))
            )

            rels = data["qrels"].get(
                q,
                {},
            )

            nc = ndcg_at_k(
                c,
                rels,
                EVAL_K,
            )

            no = ndcg_at_k(
                o,
                rels,
                EVAL_K,
            )

            nm = ndcg_at_k(
                m,
                rels,
                EVAL_K,
            )

            nd_clean.append(nc)
            nd_ord.append(no)
            nd_mc.append(nm)

            a = prefix_docs(
                data["runs"]["splade_ensemble"][q],
                L,
            )

            b = prefix_docs(
                data["runs"]["splade_self"][q],
                L,
            )

            rv = rbo_finite(
                a,
                b,
                p=RBO_P,
                depth=L,
            )

            rbo_vals.append(rv)

            if L == PRIMARY_DEPTH:
                perq_rows.append(
                    {
                        "dataset": ds_name,
                        "qid": q,
                        "depth": L,
                        "clean_ndcg": nc,
                        "ordinary_attacked_ndcg": no,
                        "mc_attacked_ndcg": nm,
                        "ordinary_delta_clean": no - nc,
                        "mc_delta_clean": nm - nc,
                        "mc_minus_ordinary": nm - no,
                        "ordinary_order_preserved": int(o == c),
                        "ordinary_set_preserved": int(set(o) == set(c)),
                        "mc_order_preserved": int(m == c),
                        "mc_set_preserved": int(set(m) == set(c)),
                        "splade_family_rbo": rv,
                    }
                )

        summary_rows.append(
            {
                "dataset": ds_name,
                "depth": L,
                "n_queries": len(qids),
                "mean_splade_family_rbo": float(
                    np.mean(rbo_vals)
                ),
                "clean_splade_family_mass": 0.5,
                "ordinary_attacked_splade_family_mass": 2.0 / 3.0,
                "mc_attacked_splade_family_mass": 0.5,
                "ordinary_exact_order_preservation": float(
                    np.mean(ord_order)
                ),
                "ordinary_exact_set_preservation": float(
                    np.mean(ord_set)
                ),
                "mc_exact_order_preservation": float(
                    np.mean(mc_order)
                ),
                "mc_exact_set_preservation": float(
                    np.mean(mc_set)
                ),
                "clean_ndcg": float(
                    np.mean(nd_clean)
                ),
                "ordinary_attacked_ndcg": float(
                    np.mean(nd_ord)
                ),
                "mc_attacked_ndcg": float(
                    np.mean(nd_mc)
                ),
                "ordinary_delta_clean": float(
                    np.mean(
                        np.asarray(nd_ord)
                        - np.asarray(nd_clean)
                    )
                ),
                "mc_delta_clean": float(
                    np.mean(
                        np.asarray(nd_mc)
                        - np.asarray(nd_clean)
                    )
                ),
                "mc_minus_ordinary": float(
                    np.mean(
                        np.asarray(nd_mc)
                        - np.asarray(nd_ord)
                    )
                ),
            }
        )

        rbo_rows.append(
            {
                "dataset": ds_name,
                "depth": L,
                "mean_rbo": float(
                    np.mean(rbo_vals)
                ),
                "median_rbo": float(
                    np.median(rbo_vals)
                ),
            }
        )


summary_df = pd.DataFrame(
    summary_rows
)

perq_df = pd.DataFrame(
    perq_rows
)

rbo_df = pd.DataFrame(
    rbo_rows
)

assert max_clean_reduction_diff <= 1e-12, max_clean_reduction_diff
assert max_undergroup_diff <= 1e-12, max_undergroup_diff

summary_df.to_csv(
    OUT / "splade_family_all_depths_v43closure.csv",
    index=False,
)

perq_df.to_csv(
    OUT / "splade_family_per_query_primary_v43closure.csv",
    index=False,
)

rbo_df.to_csv(
    OUT / "splade_family_pairwise_rbo_v43closure.csv",
    index=False,
)

print(
    "Clean singleton MC reduction max score diff:",
    f"{max_clean_reduction_diff:.3e}",
)

print(
    "Under-grouping vs ordinary max score diff:",
    f"{max_undergroup_diff:.3e}",
)

display(summary_df)


# ============================================================
# 8. FOUR PLANNED PAIRED TESTS AT DEPTH 50
# ============================================================

primary = perq_df.copy()
raw_tests = []

for ds_name in DATASETS_TO_RUN:
    d = primary[
        primary.dataset == ds_name
    ].copy()

    assert len(d) == EXPECTED_Q[ds_name]

    comparisons = [
        (
            "ordinary_attack_vs_clean",
            d["ordinary_delta_clean"].to_numpy(),
        ),
        (
            "mc_vs_attacked_ordinary",
            d["mc_minus_ordinary"].to_numpy(),
        ),
    ]

    for label, diffs in comparisons:
        ci = paired_bootstrap_ci(diffs)

        raw_tests.append(
            {
                "dataset": ds_name,
                "comparison": label,
                "mean_delta": float(
                    np.mean(diffs)
                ),
                "ci_low": float(ci[0]),
                "ci_high": float(ci[1]),
                "p_raw": wilcoxon_safe(diffs),
                "rank_biserial": rank_biserial(diffs),
                "n_queries": len(diffs),
            }
        )


stats_df = pd.DataFrame(
    raw_tests
)

assert len(stats_df) == 4

stats_df["p_holm"] = holm_adjust(
    stats_df["p_raw"].to_numpy()
)

stats_df["significant_holm_0.05"] = (
    stats_df["p_holm"] < 0.05
)

stats_df.to_csv(
    OUT / "splade_family_primary_stats_v43closure.csv",
    index=False,
)

display(stats_df)


# ============================================================
# 9. PAPER-FACING TABLE
# ============================================================

prim_summary = summary_df[
    summary_df.depth == PRIMARY_DEPTH
].copy()

rows = []

for _, s in prim_summary.iterrows():
    ds = s["dataset"]

    t_ord = stats_df[
        (stats_df.dataset == ds)
        & (
            stats_df.comparison
            == "ordinary_attack_vs_clean"
        )
    ].iloc[0]

    t_mc = stats_df[
        (stats_df.dataset == ds)
        & (
            stats_df.comparison
            == "mc_vs_attacked_ordinary"
        )
    ].iloc[0]

    rows.append(
        {
            "dataset": ds,
            "mean_RBO": s[
                "mean_splade_family_rbo"
            ],
            "ordinary_family_mass": s[
                "ordinary_attacked_splade_family_mass"
            ],
            "MC_family_mass": s[
                "mc_attacked_splade_family_mass"
            ],
            "ordinary_set_preservation": s[
                "ordinary_exact_set_preservation"
            ],
            "MC_set_preservation": s[
                "mc_exact_set_preservation"
            ],
            "ordinary_delta_clean": s[
                "ordinary_delta_clean"
            ],
            "MC_delta_clean": s[
                "mc_delta_clean"
            ],
            "MC_minus_ordinary": t_mc[
                "mean_delta"
            ],
            "MC_minus_ordinary_CI_low": t_mc[
                "ci_low"
            ],
            "MC_minus_ordinary_CI_high": t_mc[
                "ci_high"
            ],
            "MC_minus_ordinary_Holm_p": t_mc[
                "p_holm"
            ],
            "ordinary_attack_Holm_p": t_ord[
                "p_holm"
            ],
        }
    )


paper_df = pd.DataFrame(rows)

paper_df.to_csv(
    OUT / "splade_family_paper_table_v43closure.csv",
    index=False,
)

display(paper_df)


# ============================================================
# 10. COMPARE AGAINST CURRENT V4.3 MANUSCRIPT VALUES
# ============================================================

CURRENT_PAPER = {
    "SciFact": {
        "mean_RBO": 0.797,
        "ordinary_set_preservation": 0.257,
        "MC_set_preservation": 0.393,
        "ordinary_delta_clean": 0.0001,
        "MC_delta_clean": -0.0007,
        "MC_minus_ordinary": -0.0008,
        "CI_low": -0.0100,
        "CI_high": 0.0083,
        "MC_Holm_p_display": "1.000",
        "ordinary_Holm_p_display": "1.000",
    },
    "ArguAna": {
        "mean_RBO": 0.882,
        "ordinary_set_preservation": 0.390,
        "MC_set_preservation": 0.642,
        "ordinary_delta_clean": 0.0077,
        "MC_delta_clean": -0.0016,
        "MC_minus_ordinary": -0.0094,
        "CI_low": -0.0128,
        "CI_high": -0.0060,
        "MC_Holm_p_display": "1.10e-08",
        "ordinary_Holm_p_display": "4.13e-06",
    },
}


def f3(x):
    return f"{float(x):.3f}"


def f4(x):
    return f"{float(x):+.4f}"


def fp(p):
    p = float(p)

    if p < 1e-4:
        return f"{p:.2e}"

    return f"{p:.3f}"


compare_rows = []

for _, r in paper_df.iterrows():
    ds = r["dataset"]
    old = CURRENT_PAPER[ds]

    fields = [
        (
            "mean_RBO",
            f3(old["mean_RBO"]),
            f3(r["mean_RBO"]),
        ),
        (
            "ordinary_set_preservation",
            f3(old["ordinary_set_preservation"]),
            f3(r["ordinary_set_preservation"]),
        ),
        (
            "MC_set_preservation",
            f3(old["MC_set_preservation"]),
            f3(r["MC_set_preservation"]),
        ),
        (
            "ordinary_delta_clean",
            f4(old["ordinary_delta_clean"]),
            f4(r["ordinary_delta_clean"]),
        ),
        (
            "MC_delta_clean",
            f4(old["MC_delta_clean"]),
            f4(r["MC_delta_clean"]),
        ),
        (
            "MC_minus_ordinary",
            f4(old["MC_minus_ordinary"]),
            f4(r["MC_minus_ordinary"]),
        ),
        (
            "CI_low",
            f4(old["CI_low"]),
            f4(r["MC_minus_ordinary_CI_low"]),
        ),
        (
            "CI_high",
            f4(old["CI_high"]),
            f4(r["MC_minus_ordinary_CI_high"]),
        ),
        (
            "MC_Holm_p",
            old["MC_Holm_p_display"],
            fp(r["MC_minus_ordinary_Holm_p"]),
        ),
        (
            "ordinary_Holm_p",
            old["ordinary_Holm_p_display"],
            fp(r["ordinary_attack_Holm_p"]),
        ),
    ]

    for field, old_display, new_display in fields:
        compare_rows.append(
            {
                "dataset": ds,
                "field": field,
                "current_paper_display": old_display,
                "v43_closure_display": new_display,
                "same_at_paper_precision": (
                    old_display == new_display
                ),
            }
        )


comparison_df = pd.DataFrame(
    compare_rows
)

comparison_df.to_csv(
    OUT / "current_paper_vs_v43_splade_closure.csv",
    index=False,
)

display(comparison_df)

changed = int(
    (
        ~comparison_df[
            "same_at_paper_precision"
        ]
    ).sum()
)

print(
    f"Paper-facing fields changed at displayed precision: "
    f"{changed}/{len(comparison_df)}"
)


# ============================================================
# 11. GENERATE MANUSCRIPT-READY TEXT AND LATEX TABLE
# ============================================================

def row_for(ds):
    x = paper_df[
        paper_df.dataset == ds
    ]

    assert len(x) == 1
    return x.iloc[0]


def latex_p(p):
    p = float(p)

    if p < 1e-4:
        mant, exp = f"{p:.2e}".split("e")
        return (
            f"{mant} \\\\times "
            f"10^{{{int(exp)}}}"
        )

    return f"{p:.3f}"


sf = row_for("SciFact")
ar = row_for("ArguAna")

manuscript_text = (
    "The structural result is consistent across both datasets: fixed family mass "
    f"increases exact top-10 set preservation from "
    f"{100*sf['ordinary_set_preservation']:.1f}\\\\% to "
    f"{100*sf['MC_set_preservation']:.1f}\\\\% on SciFact and from "
    f"{100*ar['ordinary_set_preservation']:.1f}\\\\% to "
    f"{100*ar['MC_set_preservation']:.1f}\\\\% on ArguAna. "
    "The effectiveness result is deliberately not summarized as a second win. "
    f"On SciFact, ordinary expansion changes nDCG@10 by only "
    f"${sf['ordinary_delta_clean']:+.4f}$ and MC-RRF differs from expanded "
    f"ordinary RRF by ${sf['MC_minus_ordinary']:+.4f}$ with a confidence interval "
    "spanning zero. "
    f"On ArguAna, however, ordinary expansion improves nDCG@10 by "
    f"${ar['ordinary_delta_clean']:+.4f}$ relative to clean, Holm "
    f"$p={latex_p(ar['ordinary_attack_Holm_p'])}$, while MC-RRF is "
    f"${ar['MC_minus_ordinary']:+.4f}$ below expanded ordinary RRF, 95\\\\% CI "
    f"$[{ar['MC_minus_ordinary_CI_low']:+.4f},"
    f"{ar['MC_minus_ordinary_CI_high']:+.4f}]$, Holm "
    f"$p={latex_p(ar['MC_minus_ordinary_Holm_p'])}$. "
    "The second SPLADE checkpoint therefore carries useful ranking variation on "
    "ArguAna even though it shares provenance and has high rank overlap with the "
    "first checkpoint."
)


table_lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{SPLADE checkpoint-family boundary experiment at depth 50.}",
    r"\begin{adjustbox}{max width=\linewidth}",
    r"\begin{tabular}{lrrrrrrr}",
    r"\toprule",
    (
        r"Dataset & Mean RBO & Ord. set & MC set & Ord. $\Delta$ & "
        r"MC $\Delta$ & MC$-$Ord. $\Delta$ [95\% CI] & Holm $p$ \\"
    ),
    r"\midrule",
]

for ds in ["SciFact", "ArguAna"]:
    r = row_for(ds)

    table_lines.append(
        f"{ds} & "
        f"{r['mean_RBO']:.3f} & "
        f"{r['ordinary_set_preservation']:.3f} & "
        f"{r['MC_set_preservation']:.3f} & "
        f"{r['ordinary_delta_clean']:+.4f} & "
        f"{r['MC_delta_clean']:+.4f} & "
        f"{r['MC_minus_ordinary']:+.4f} "
        f"[{r['MC_minus_ordinary_CI_low']:+.4f}, "
        f"{r['MC_minus_ordinary_CI_high']:+.4f}] & "
        f"${latex_p(r['MC_minus_ordinary_Holm_p'])}$ \\\\"
    )

table_lines.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{adjustbox}",
        r"\end{table}",
    ]
)

table_tex = "\n".join(table_lines)

display(
    Markdown(
        "## Regenerated interpretation\n\n"
        + manuscript_text.replace("\\\\%", "%")
    )
)

print("\nLATEX TABLE:\n")
print(table_tex)


# ============================================================
# 12. WRITE PROVENANCE/CLOSURE ARTIFACTS
# ============================================================

source_truth = {
    "frozen_v43_archive": (
        SOURCE_ARCHIVE.name
        if SOURCE_ARCHIVE is not None
        else None
    ),
    "frozen_source_hashes": source_hashes,
    "generated_selfdistil_hashes": self_run_hashes,
    "selfdistil_model": SELF_MODEL,
    "selfdistil_depth": SELF_TOPK,
    "rrf_k": RRF_K,
    "eval_k": EVAL_K,
    "depths": DEPTHS,
    "primary_depth": PRIMARY_DEPTH,
    "family_definition": (
        "Declared NAVER SPLADE CoCondenser checkpoint family; "
        "qrels are not used for family definition."
    ),
    "clean_family_mass_normalized": 0.5,
    "ordinary_expanded_family_mass_normalized": 2.0 / 3.0,
    "mc_expanded_family_mass_normalized": 0.5,
    "canonical_sources_regenerated": False,
    "selfdistil_generated_in_closure": True,
    "tie_break": (
        "descending score, then ascending document_id; "
        "full-query fallback at an unresolved retrieval-boundary tie"
    ),
}

(
    OUT
    / "SPLADE_V43_CLOSURE_PROTOCOL.json"
).write_text(
    json.dumps(
        source_truth,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

(
    OUT
    / "SPLADE_V43_MANUSCRIPT_TEXT.tex"
).write_text(
    manuscript_text
    + "\n\n"
    + table_tex
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 13. FINAL FAIL-LOUD AUDIT AND SELF-VERIFYING PACKAGE
# ============================================================

required_outputs = [
    "splade_family_all_depths_v43closure.csv",
    "splade_family_per_query_primary_v43closure.csv",
    "splade_family_pairwise_rbo_v43closure.csv",
    "splade_family_primary_stats_v43closure.csv",
    "splade_family_paper_table_v43closure.csv",
    "current_paper_vs_v43_splade_closure.csv",
    "selfdistil_generation_log.csv",
    "SPLADE_V43_CLOSURE_PROTOCOL.json",
    "SPLADE_V43_MANUSCRIPT_TEXT.tex",
]

assert max_clean_reduction_diff <= 1e-12
assert max_undergroup_diff <= 1e-12

assert set(
    paper_df.dataset
) == set(
    DATASETS_TO_RUN
)

assert np.allclose(
    paper_df[
        "MC_family_mass"
    ].to_numpy(),
    0.5,
)

assert np.allclose(
    paper_df[
        "ordinary_family_mass"
    ].to_numpy(),
    2.0 / 3.0,
)

for fn in required_outputs:
    assert (
        OUT / fn
    ).exists(), fn

for ds_name in DATASETS_TO_RUN:
    run_path = (
        RUN_OUT
        / f"{ds_name}_splade_selfdistil_top500.json"
    )
    assert run_path.exists(), run_path


closure_manifest = {
    "source_archive": (
        SOURCE_ARCHIVE.name
        if SOURCE_ARCHIVE is not None
        else None
    ),
    "frozen_source_hashes": source_hashes,
    "generated_selfdistil_hashes": self_run_hashes,
    "files": {},
}

for p in sorted(
    OUT.rglob("*")
):
    if (
        p.is_file()
        and p.name != "CLOSURE_MANIFEST.json"
    ):
        closure_manifest[
            "files"
        ][
            str(
                p.relative_to(OUT)
            )
        ] = sha256_file(p)


closure_manifest_path = (
    OUT
    / "CLOSURE_MANIFEST.json"
)

closure_manifest_path.write_text(
    json.dumps(
        closure_manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

check = json.loads(
    closure_manifest_path.read_text(
        encoding="utf-8"
    )
)

for rel, expected in check[
    "files"
].items():
    actual = sha256_file(
        OUT / rel
    )

    assert actual == expected, (
        rel,
        actual,
        expected,
    )

print(
    "CLOSURE MANIFEST SELF-VERIFY: PASS",
    len(check["files"]),
    "files",
)


package_path = (
    WORK_ROOT
    / "InvariantRRF_V43_SPLADE_TwoCheckpoint_Closure.zip"
)

if package_path.exists():
    package_path.unlink()

with zipfile.ZipFile(
    package_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(
        OUT.rglob("*")
    ):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    Path(OUT.name)
                    / p.relative_to(OUT)
                ),
            )


print("\nFINAL AUDIT: PASS")
print("Frozen BM25 reused: PASS")
print("Frozen EnsembleDistil reused: PASS")
print("Only SelfDistil generated: PASS")
print("Clean MC-RRF singleton reduction: PASS")
print("Under-grouping reproduces ordinary RRF: PASS")
print("Fixed SPLADE family mass after expansion: PASS")

print(
    "\nPaper-facing fields changed:",
    changed,
    "/",
    len(comparison_df),
)

print("\nFINAL PACKAGE:")
print(package_path)


## What to send back after the run

Please send either:

- `InvariantRRF_V43_SPLADE_TwoCheckpoint_Closure.zip`, or
- the final output from the notebook.

The key line is:

`Paper-facing fields changed: X / 20`

If it is `0 / 20`, the current SPLADE boundary table is reproduced from the frozen v4.3 sources and this issue can be closed. If anything changes, use the regenerated values from this closure notebook.
